# Kinlens KNN Performance Benchmark

This notebook compares **Kinlens** with **scikit-learn's `KNeighborsClassifier`** on the Social Network Ads dataset.

The comparison uses:

- Euclidean distance
- `k = 3`
- Inverse-distance voting for Kinlens
- Distance weighting for scikit-learn
- The same 75/25 stratified train/test split
- Accuracy, F1 score, recall, and prediction agreement

## 1. Setup

The benchmark uses the project package directly. No Kaggle download or network access is required because the dataset is stored in the repository.

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score as accuracyScore,
    f1_score as f1Score,
    recall_score as recallScore,
)
from sklearn.model_selection import train_test_split as trainTestSplit
from sklearn.neighbors import KNeighborsClassifier

from kinlens.classifier import Classifier
from kinlens.datasets import Dataset
from kinlens.metrics import Euclidean
from kinlens.search import BruteForseSearch
from kinlens.utils.dataset import DatasetBuilder

## 2. Load the Dataset

`Age` and `EstimatedSalary` are selected as features. `Purchased` is the target label.

In [25]:
sampleCandidates = [
    Path("samples/social-network-ads.csv"),
    Path("src/kinlens/notebooks/samples/social-network-ads.csv"),
]

samplePath = next(path for path in sampleCandidates if path.exists())

datasetBuilder = DatasetBuilder(
    path=samplePath,
    features=["Age", "EstimatedSalary"],
    label="Purchased",
)
kinlensDataset = datasetBuilder.build()

print(f"Dataset: {len(kinlensDataset.x)} rows")
print(f"Features: {kinlensDataset.features}")
print(f"Classes: {kinlensDataset.classes}")

Dataset: 400 rows
Features: ['Age', 'EstimatedSalary']
Classes: ['0', '1']


Kinlens encodes the target labels and exposes numeric features in `x` and encoded labels in `y`.

In [26]:
pd.DataFrame(
    kinlensDataset.x,
    columns=kinlensDataset.features,
).assign(Purchased=kinlensDataset.y).head(10)

,Age,EstimatedSalary,Purchased
0,19.0,19000.0,0
1,35.0,20000.0,0
2,26.0,43000.0,0
3,27.0,57000.0,0
4,19.0,76000.0,0
5,27.0,58000.0,0
6,27.0,84000.0,0
7,32.0,150000.0,1
8,25.0,33000.0,0
9,35.0,65000.0,0


## 3. Neighbor Search Example

A single query is used to verify that both implementations find the same nearest neighbors before running the benchmark.

In [27]:
bruteSearch = BruteForseSearch(metric=Euclidean()).fit(kinlensDataset)
query = np.array([1.3, 5.3])
kinlensNeighbors = bruteSearch.findNeighbors(query=query, k=3)

pd.DataFrame(
    {
        "Rank": range(1, 4),
        "Features": [kinlensDataset.x[index].tolist() for index in kinlensNeighbors.indices],
        "Label": [int(kinlensDataset.y[index]) for index in kinlensNeighbors.indices],
        "Distance": kinlensNeighbors.distances,
    }
)

,Rank,Features,Label,Distance
0,1,"[26.0, 15000.0]",0,14994.720344
1,2,"[26.0, 15000.0]",0,14994.720344
2,3,"[30.0, 15000.0]",0,14994.727466


In [28]:
sklearnNeighbors = KNeighborsClassifier(
    n_neighbors=3,
    weights="distance",
    metric="euclidean",
)
sklearnNeighbors.fit(kinlensDataset.x, kinlensDataset.y)

sklearnDistances, sklearnIndices = sklearnNeighbors.kneighbors(query.reshape(1, -1))

pd.DataFrame(
    {
        "Rank": range(1, 4),
        "Features": [kinlensDataset.x[index].tolist() for index in sklearnIndices[0]],
        "Label": [int(kinlensDataset.y[index]) for index in sklearnIndices[0]],
        "Distance": sklearnDistances[0],
    }
)

,Rank,Features,Label,Distance
0,1,"[26.0, 15000.0]",0,14994.720344
1,2,"[26.0, 15000.0]",0,14994.720344
2,3,"[30.0, 15000.0]",0,14994.727466


## 4. Comparative Benchmark

Both classifiers receive the exact same training and test rows. The split is stratified and reproducible with `random_state=42`.

Scikit-learn is configured with distance weighting to match Kinlens's default inverse-distance voting.

In [29]:
xTrain, xTest, yTrain, yTest = trainTestSplit(
    kinlensDataset.x,
    kinlensDataset.y,
    test_size=0.25,
    random_state=42,
    stratify=kinlensDataset.y,
)

trainDataset = Dataset(
    x=xTrain,
    y=yTrain,
    features=kinlensDataset.features,
    classes=kinlensDataset.classes,
    name=kinlensDataset.name,
    targetEncodingMap=kinlensDataset.targetEncodingMap,
)

print(f"Training rows: {len(xTrain)}")
print(f"Test rows: {len(xTest)}")

Training rows: 300
Test rows: 100


In [30]:
sklearnKnn = KNeighborsClassifier(
    n_neighbors=3,
    weights="distance",
    metric="euclidean",
).fit(xTrain, yTrain)

kinlensKnn = Classifier(k=3).fit(trainDataset)

In [31]:
sklearnPredictions = sklearnKnn.predict(xTest)

kinlensPredictions = np.array([
    trainDataset.targetEncodingMap[kinlensKnn.predict(queryRow)]
    for queryRow in xTest
])

## 5. Metrics

- **Accuracy** measures overall correct predictions.
- **F1** balances precision and recall.
- **Recall** measures the proportion of actual positive samples identified correctly.

In [32]:
comparisonTable = pd.DataFrame(
    {
        "Metric": ["Accuracy", "F1", "Recall"],
        "scikit-learn KNN": [
            accuracyScore(yTest, sklearnPredictions),
            f1Score(yTest, sklearnPredictions),
            recallScore(yTest, sklearnPredictions),
        ],
        "Kinlens KNN": [
            accuracyScore(yTest, kinlensPredictions),
            f1Score(yTest, kinlensPredictions),
            recallScore(yTest, kinlensPredictions),
        ],
    }
).set_index("Metric")

displayTable = comparisonTable.copy()
for columnName in ["scikit-learn KNN", "Kinlens KNN"]:
    displayTable[columnName] = displayTable[columnName].map(lambda value: f"{value:.2%}")

displayTable

,scikit-learn KNN,Kinlens KNN
Metric,,
Accuracy,75.00%,75.00%
F1,60.32%,60.32%
Recall,52.78%,52.78%


In [33]:
predictionAgreement = np.mean(sklearnPredictions == kinlensPredictions)
print(f"Prediction agreement: {predictionAgreement:.2%}")

print("\nResult summary:")
print(displayTable.to_string())

Prediction agreement: 100.00%

Result summary:
         scikit-learn KNN Kinlens KNN
Metric                               
Accuracy           75.00%      75.00%
F1                 60.32%      60.32%
Recall             52.78%      52.78%


## 6. Results

| Metric | scikit-learn KNN | Kinlens KNN |
|---|---:|---:|
| Accuracy | 75.00% | 75.00% |
| F1 | 60.32% | 60.32% |
| Recall | 52.78% | 52.78% |

Both classifiers produce identical predictions on all 100 test rows, resulting in 100% prediction agreement.

This benchmark confirms that Kinlens matches scikit-learn for this dataset and configuration. The result also demonstrates that Kinlens's explainable search, voting, and prediction pipeline can reproduce standard KNN behavior.